In [1]:
import numpy as np
import matplotlib.pyplot as plt

import sys, os
from pathlib import Path

gps_path = Path(os.path.abspath("")).resolve().parent
sys.path.insert(len(sys.path), str(gps_path.resolve()))

from gps import gps_receiver

In [2]:
from typing import List
from dataclasses import dataclass, fields

In [3]:
fs = 4.092e6

data = np.fromfile("../data/1/gpssim.ci16", dtype=np.int8, count=int(fs*60))
data = data.astype(np.float32).view(np.complex64)

In [4]:
rx = gps_receiver.GpsReceiver(fs)
rx.process(data)
del data

frames_ref = rx.dump_frames()
frames = list(frames_ref)

[Acquisition] Searching 2.00ms period, N=8192
[Acquisition] 499.5Hz frequency bins
[Acquisition] SV: 3, Doppler shift: -999.02Hz, Code sample phase: 2059/4092, SNR: 12.5
[Acquisition] SV: 4, Doppler shift: 2997.07Hz, Code sample phase: 3753/4092, SNR: 8.9
[Acquisition] SV: 9, Doppler shift: 3496.58Hz, Code sample phase: 1208/4092, SNR: 9.5
[Acquisition] SV: 16, Doppler shift: 2497.56Hz, Code sample phase: 2591/4092, SNR: 17.0
[Acquisition] SV: 26, Doppler shift: 0.00Hz, Code sample phase: 1866/4092, SNR: 19.6
[Acquisition] SV: 27, Doppler shift: 3496.58Hz, Code sample phase: 2376/4092, SNR: 7.3
[Acquisition] SV: 28, Doppler shift: -1998.05Hz, Code sample phase: 1132/4092, SNR: 14.9
[Acquisition] SV: 29, Doppler shift: -499.51Hz, Code sample phase: 3336/4092, SNR: 7.4
[Acquisition] SV: 31, Doppler shift: -999.02Hz, Code sample phase: 3774/4092, SNR: 12.2
[Acquisition] SV: 32, Doppler shift: -3496.58Hz, Code sample phase: 3066/4092, SNR: 7.3
[Acquisition] Fine freq: -1061.7 Hz, +-62.4 Hz

In [5]:
def group_frames(frame_list: List[gps_receiver.Subframe]):
    groups = {}

    for f in frame_list:
        if f.tow in groups.keys():
            groups[f.tow].append(f)
        else:
            groups[f.tow] = [f]

    return groups

frames_by_tow = group_frames(frames)

In [6]:
frames_by_sv = {}

for f in frames:
    if f.sv not in frames_by_sv.keys():
        frames_by_sv[f.sv] = [f]
    else:
        frames_by_sv[f.sv].append(f)

In [37]:
# Constants from GPS ICD
mu = 3.986005e14 # m^3/s^2
F = -4.442807633e-10 # s/sqrt(m)
c = 2.99792458e8 # m/s
gps_pi = 3.1415926535898
Omega_dot_ref = -2.6e-9 # rads/sec
Omega_dot_e = 7.2921151467e-5 # rads/sec, WGS84 earth rotation rate

In [69]:
@dataclass
class SvParams:
    sv: int
    tow: int = None
    week: int = None
    iodc: int = None
    t_gd: float = None
    t_oc: int = None
    a_f2: float = None
    a_f1: float = None
    a_f0: float = None
    iode: int = None
    c_rs: float = None
    delta_n: float = None
    m_0: float = None
    c_uc: float = None
    e: float = None
    c_us: float = None
    sqrt_a: float = None
    t_oe: float = None
    c_ic: float = None
    Omega_0: float = None
    c_is: float = None
    i_0: float = None
    c_rc: float = None
    omega: float = None
    Omega_dot: float = None
    idot: float = None
    iode: float = None

    def update(self, frame: gps_receiver.Subframe):
        if frame.sv != self.sv:
            return

        for field in fields(self):
            if field.name in dir(frame):
                setattr(self, field.name, getattr(frame, field.name))

    def mean_motion(self):
        return np.sqrt(mu / (self.sqrt_a**2)**3) + self.delta_n

    # t is GPS system time corrected for transit time
    def eccentric_anomaly(self, t: float):
        t_k = t - self.t_oe

        if t_k > 302400:
            t_k -= 604800
        elif t_k < -302400:
            t_k += 604800

        M_k = self.m_0 + self.mean_motion() * t_k

        E = M_k

        for _ in range(3):
            dE = (M_k - E + self.e * np.sin(E))/(1 - self.e*np.cos(E))
            E = E + dE

        return E

In [74]:
class Sv:
    def __init__(self, frames: List[gps_receiver.Subframe]):
        self.sv = frames[0].sv
        self.p = SvParams(sv=self.sv)

        # Latest of each type of frame
        self.latest = [None, None, None, None, None]

        self.update(frames)

    def update(self, frames: List[gps_receiver.Subframe]):
        for f in frames:
            if f.sv != self.sv:
                continue

            if self.latest[f.id - 1] == None or self.latest[f.id - 1].tow < f.tow:
                self.latest[f.id - 1] = f
                self.p.update(f)

    def corrected_time(self, t_c):
        dt_r = F * self.p.e * self.p.sqrt_a * np.sin(self.p.eccentric_anomaly(t_c))
        dt_sv = (
            self.p.a_f0
            + self.p.a_f1 * (t_c - self.p.t_oc)
            + self.p.a_f2 * (t_c - self.p.t_oc) ** 2
            + dt_r
        )

        return t_c - dt_sv

    def sv_position(self, t_rel):
        t_c = self.p.tow - t_rel
        print(t_c)

        if t_c - self.p.t_oe > 302400:
            t_c = t_c - 604800
        elif t_c - self.p.t_oe < -302400:
            t_c = t_c + 604800

        t_c = t_c - self.p.t_oe
        print(t_c, self.p.t_oe)

        E = self.p.eccentric_anomaly(t_c)
        t = self.corrected_time(t_c)
        print(t)

        print()

        # True anomaly
        nu_1 = np.arccos((np.cos(E) - self.p.e) / (1 - self.p.e**2 * np.cos(E)))
        nu_2 = np.arcsin(
            (np.sqrt(1 - self.p.e**2) * np.sin(E)) / (1 - self.p.e * np.cos(E))
        )
        nu = nu_1 * np.sign(nu_2)

        # Argument of latitude
        phi = nu + self.p.omega

        # 2nd harmonic corrections
        d_phi = self.p.c_us * np.sin(2 * phi) + self.p.c_uc * np.cos(2 * phi)
        d_r = self.p.c_rs * np.sin(2 * phi) + self.p.c_rc * np.cos(2 * phi)
        d_i = self.p.c_is * np.sin(2 * phi) + self.p.c_ic * np.cos(2 * phi)

        # Corrected params
        phi = phi + d_phi
        r = self.p.sqrt_a**2 * (1 - self.p.e * np.cos(E)) + d_r
        i = self.p.i_0 + self.p.idot * t + d_i

        # Positions in orbital plane
        x_p = r * np.cos(phi)
        y_p = r * np.sin(phi)

        # Corrected longitude of ascending node
        Omega = (
            self.p.Omega_0
            + (self.p.Omega_dot - Omega_dot_e) * t
            - Omega_dot_e * self.p.t_oe
        )

        # ECEF (earth fixed) coords
        x = x_p * np.cos(Omega) - y_p * np.sin(Omega)
        y = x_p * np.sin(Omega) + y_p * np.cos(Omega)
        z = y_p * np.sin(i)

        # TODO: calculate SV velocity, not really needed right now

        return [x, y, z]

In [75]:
group: List[gps_receiver.Subframe] = frames_by_tow[345600]
ref_frame = group[0]

sv_positions = []
pseudoranges = []

for i in range(1, len(group)):
    f = group[i]
    t_local = (group[i].position - ref_frame.position)/fs

    t_transit = t_local + 0.075

    sv = Sv(frames_by_sv[f.sv])

    sv_positions.append(sv.sv_position(t_transit))

    print(sv.sv, sv_positions[-1])

    # GPS time at transmission corrected by transit time
    t_c = ref_frame.tow - t_transit

    # t_tropo = 0
    # t_iono = 0

    # pseudorange constant approximately 75ms
    r = 299792458 * t_transit

    pseudoranges.append(r)
    # TODO: correct pseudoranges for troposphere and ionosphere

345599.9147919918
-0.08520800818223506 345600
-0.08514400421555181

9 [11039448.729601875, 24077548.908645317, 3325360.2534823543]
345599.92913000967
-0.0708699903334491 345600
-0.07041354987420403

16 [22947381.689871244, -12654598.921369875, 673486.1032913314]
345599.92477346346
-0.0752265365445055 345600
-0.07521886518184262

28 [26223775.660748266, -4109578.43733019, 5772925.2930875495]
345599.91831207264
-0.08168792736250907 345600
-0.08107878319475341

29 [26120519.472806264, 4448165.605135554, 3359847.524978842]
345599.91924605495
-0.08075394504703581 345600
-0.08019861138610407

32 [25964813.919579506, -4639973.877035487, -6169180.6512953965]


Reference sv positions:

SV 9 [12300521.850039, -9850509.176040, -21141090.416022]
SV 16 [-22352748.956905, 12911766.246194, -6055779.961539]
SV 28 [18554628.608471, 1668444.966275, 18926446.540396]
SV 29 [-23373271.436217, 2280972.791292, -12535569.811159]
SV 32 [-19269158.610949, -14531194.231824, -11809604.648607]

SV 3 [-11999740.155682, -9558653.706877, 21697588.030084]
SV 4 [5075847.747261, 25835095.109249, 2883676.636369]
SV 26 [9216403.794850, -21062076.097253, -13183324.576334]
SV 27 [18298201.977208, -7733894.598181, 17608838.113404]
SV 31 [20470912.978652, -16453688.293285, -2941747.650266]

Expected position: 1121575.5,  -4835955.6,   3991116.0

In [68]:
def solve_position(r_sv, p):
    # Position estimate
    r_est = np.array([0, 0, 0], dtype=np.float32)

    # Geometric range
    rho = np.zeros(len(p))


    # Should never need more than 3-5
    for _ in range(10):
        A = np.zeros((len(p), 4))

        for i in range(len(p)):
            rho[i] = np.linalg.norm(r_sv[i] - r_est)

            A[i, 0] = -(r_sv[i][0] - r_est[0]) / rho[i]
            A[i, 1] = -(r_sv[i][1] - r_est[1]) / rho[i]
            A[i, 2] = -(r_sv[i][2] - r_est[2]) / rho[i]
            A[i, 3] = 1

        # Least squares
        x = np.linalg.inv(A.T @ A) @ A.T @ (p - rho)

        r_est += x[:3]

        if np.linalg.norm(x[:3]) < 1:
            break

    return r_est

expected = [1121575.5, -4835955.6, 3991116.0]
actual = solve_position(sv_positions, pseudoranges)

print(f"Expected: {expected}")
print(f"Actual: {actual}")

Expected: [1121575.5, -4835955.6, 3991116.0]
Actual: [-4868212.  -4502249.5  3819714.5]
